# Regularization: Ridge & Lasso

**Companion lesson:** https://ml-viz.vercel.app/courses/linear-regression/03-regularization

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Overfitting with correlated features

We build a dataset where only 3 of 20 features matter, then watch OLS, Ridge, and Lasso handle it.

In [ ]:
n, d = 60, 20
X = np.random.randn(n, d)
true_w = np.zeros(d); true_w[:3] = [3.0, -2.0, 1.5]
y = X @ true_w + 0.5 * np.random.randn(n)

# closed-form ridge: w = (X'X + lam*I)^-1 X'y   (lam=0 -> OLS)
def ridge(X, y, lam):
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)

print("OLS weights (first 6):", np.round(ridge(X, y, 0)[:6], 2))
print("Ridge λ=10  (first 6):", np.round(ridge(X, y, 10)[:6], 2))

## Lasso via proximal gradient (ISTA)

The L1 penalty has no closed form, but coordinate-wise soft-thresholding solves it.

In [ ]:
def soft(z, t): return np.sign(z) * np.maximum(np.abs(z) - t, 0)

def lasso(X, y, lam, iters=500):
    w = np.zeros(X.shape[1]); L = np.linalg.norm(X, 2) ** 2
    for _ in range(iters):
        w = soft(w - X.T @ (X @ w - y) / L, lam / L)
    return w

w_lasso = lasso(X, y, lam=15)
print("Lasso non-zero weights:", np.flatnonzero(np.abs(w_lasso) > 1e-6))
print("values:", np.round(w_lasso[np.abs(w_lasso) > 1e-6], 2))

## Regularization paths

Watch every weight as λ sweeps — Ridge shrinks smoothly, Lasso snaps weights to exactly zero.

In [ ]:
lams = np.logspace(-2, 3, 40)
ridge_path = np.array([ridge(X, y, l) for l in lams])
lasso_path = np.array([lasso(X, y, l) for l in lams])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, path, name in [(axes[0], ridge_path, 'Ridge'), (axes[1], lasso_path, 'Lasso')]:
    for j in range(d):
        ax.plot(lams, path[:, j], color='#6366f1' if j < 3 else '#475569', lw=1.5 if j < 3 else 0.7)
    ax.set_xscale('log'); ax.set_xlabel('λ'); ax.set_title(f'{name} path')
axes[0].set_ylabel('weight value')
plt.tight_layout(); plt.show()
# purple = the 3 true features; gray = the 17 noise features

**Try it:** raise the noise level to 2.0, or make features correlated with `X[:, 3] = X[:, 0] + 0.01*np.random.randn(n)`. Watch what OLS does to those two weights vs. Ridge.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Soft thresholding

The operator at the heart of lasso (and of ISTA above) shrinks every value toward zero by $t$ — and snaps anything within $t$ of zero to **exactly zero**:

$$S_t(x) = \text{sign}(x) \cdot \max(|x| - t, \, 0)$$

That hard zero is where lasso's sparsity comes from (ridge only ever scales weights down — it never zeroes them). Implement it vectorized.

In [ ]:
def soft_threshold(x, t):
    """Shrink x toward 0 by t; values within t of 0 become exactly 0."""
    x = np.asarray(x, dtype=float)

    # TODO(you): sign(x) * max(|x| - t, 0) — keep it vectorized
    # (hint: np.sign, np.maximum, np.abs)
    return ...

In [ ]:
# Checks — run me
assert np.allclose(soft_threshold(3.0, 1.0), 2.0), "shrink positive values toward 0"
assert np.allclose(soft_threshold(-3.0, 1.0), -2.0), "shrink negative values toward 0"
assert np.allclose(soft_threshold(0.4, 1.0), 0.0), "inside the threshold -> exactly 0 (sparsity!)"
assert np.allclose(soft_threshold(np.array([-2.0, -0.5, 0.0, 0.5, 2.0]), 1.0),
                   [-1.0, 0.0, 0.0, 0.0, 1.0]), "works on arrays"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def soft_threshold(x, t):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.maximum(np.abs(x) - t, 0.0)
```

</details>

### Exercise 2 — One ISTA step

ISTA alternates a plain gradient step on the squared error with a soft-threshold whose level is $\eta\lambda$:

$$\mathbf{w} \leftarrow S_{\eta\lambda}\!\big(\mathbf{w} - \eta \, X^\top(X\mathbf{w} - \mathbf{y})\big)$$

Implement one step. With an **identity design matrix** ($X = I$) and $\eta = 1$, a single step from $\mathbf{w} = 0$ lands *exactly* on the lasso solution $S_\lambda(\mathbf{y})$ — the checks verify that, the exact zero it creates, and that iterating from elsewhere converges to the same point.

In [ ]:
def ista_step(w, X, y, lam, lr):
    """One ISTA update for the lasso objective 0.5*||Xw - y||^2 + lam*||w||_1."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): the squared-error gradient X^T (X w - y)
    grad = ...

    # TODO(you): gradient step, then soft-threshold at level lr * lam
    return ...

In [ ]:
# Checks — run me
y_t = np.array([3.0, -0.5, 1.5])
I = np.eye(3)

w1 = ista_step(np.zeros(3), I, y_t, lam=1.0, lr=1.0)
assert np.allclose(w1, soft_threshold(y_t, 1.0)), \
    "identity design, lr = 1: one step from 0 lands exactly on soft_threshold(y, lam)"
assert w1[1] == 0.0, "the small coefficient is zeroed exactly — lasso sparsity"

w = np.array([10.0, -10.0, 10.0])
for _ in range(50):
    w = ista_step(w, I, y_t, lam=1.0, lr=0.5)
assert np.allclose(w, soft_threshold(y_t, 1.0), atol=1e-8), "ISTA converges to the lasso solution"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ista_step(w, X, y, lam, lr):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    grad = X.T @ (X @ w - y)
    return soft_threshold(w - lr * grad, lr * lam)
```

</details>